## Домашнее задание по неделе 4

Как было рассказано на лекции, стохастический градиентый спуск сходится быстрее, чем полный, хотя и менее стабильно. В этом задании вам предлагается реализовать стохастический градиентный спуск и сравнить его с точным вычислением весов линейной модели по скорости работы и значению метрики качества.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

np.random.seed(0)

warnings.filterwarnings('ignore')
%matplotlib inline

### Задание 0

Реализуйте класс ```LinearRegressionSGD``` c обучением и и применением линейной регрессии, построенной с помощью стохастического градиентного спуска, с заданным интерфейсом.

In [37]:
from sklearn.base import BaseEstimator

class LinearRegressionSGD(BaseEstimator):
    def __init__(self, epsilon=1e-4, max_steps=100, w0=None, alpha=1e-4):
        """
        epsilon: разница для нормы изменения весов
        max_steps: максимальное количество шагов в градиентном спуске
        w0: np.array (d,) - начальные веса
        alpha: шаг обучения
        """
        self.epsilon = epsilon
        self.max_steps = max_steps
        self.w0 = w0
        self.alpha = alpha
        self.w = None
        self.w_history = []

    def fit(self, X, y):
        """
        X: np.array (l, d)
        y: np.array (l)
        ---
        output: self
        """
        ## На каждом шаге градиентного спуска веса необходимо добавлять в w_history
        X = np.asarray(X)
        y = np.asarray(y)
        l, d = X.shape

        if self.w0 is None:
            self.w0 = np.zeros(d)
        
        self.w = self.w0

        for _ in range(self.max_steps):
            self.w_history.append(self.w)

            new_w = self.w - self.alpha * self.calc_gradient(X, y)

            if np.linalg.norm(new_w - self.w) < self.epsilon:
                self.w = new_w
                break

            self.w = new_w

        return self

    def predict(self, X):
        """
        X: np.array (l, d)
        ---
        output: np.array (l)
        """
        l, d = X.shape

        return np.dot(X, self.w)

    def calc_gradient(self, X, y):
        """
        X: np.array (l, d)
        y: np.array (l)
        ---
        output: np.array (d)
        """
        l, d = X.shape
        gradient = []

        i = np.random.randint(l)

        for j in range(d):
            L = 2 * X[i][j] * (np.dot(X[i], self.w) - y[i])
            gradient.append(L)

        return np.array(gradient)

Проверять работу мы будем на имеющемся в sklearn наборе данных.
Возьмем стандартный [датасет](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) со стоимостью жилья в различных районах Калифорнии в 1990 году.  Датасет содержит информацию о средних ценах на жилье в районе и какие-то параметры района: средний возраст домов, среднее число комнат, население

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

In [59]:
X_test.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
20303,5.2783,12.0,4.466019,0.980583,405.0,3.932039,34.16,-119.18
16966,3.9722,27.0,4.619271,1.096357,1877.0,2.205640,37.55,-122.31
10623,4.5094,12.0,4.426380,1.092025,1913.0,1.956033,33.67,-117.77
6146,3.1034,29.0,4.597222,1.037037,2013.0,4.659722,34.11,-117.95
2208,4.6726,6.0,5.730303,1.033333,969.0,2.936364,36.81,-119.87


### Задание 1

Метрикой качества в нашей задаче будет MAPE - Mean Absolute Percentage Error. Реализуйте её с заданным интефейсом и вычислите
```MAPE(y_test, y_0)```, где ```y_0 = (mean(y_test), mean(y_test), ...)```

In [42]:
def MAPE(y_true, y_pred):
    """
        y_true: np.array (l)
        y_pred: np.array (l)
        ---
        output: float [0, +inf)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    sum = 0
    for i in range(len(y_true)):
        if y_true[i] != 0:
            mape = np.abs((y_true[i] - y_pred[i]) / y_true[i])
            sum += mape
    
    ans = float(sum * 100 / len(y_true))
    return ans

In [ ]:
y_0 = np.ones(len(y_test)) * np.mean(y_test)

MAPE(np.asarray(y_test), y_0)

62.212084748204994

### Задание 2

Обучите ```LinearRegressionSGD``` с базовыми параметрами на тренировочном наборе данных (```X_train```, ```y_train```), сделайте предсказание на тестовых данных ```X_test```, записав результат в переменную ```y_pred_sgd```  и вычислите ошибку MAPE.

In [74]:
sgd = LinearRegressionSGD()
sgd.fit(X_train, y_train)

y_pred_sgd = sgd.predict(X_test)

MAPE(y_test, y_pred_sgd)

5.482722516350869e+244

### Задание 3

Вычислите веса по точной формуле, используя ```X_train``` и ```y_train```; предскажите с их помощью целевую переменную на ```X_test```, записав результат в переменную ```y_pred_lr``` и вычислите ошибку MAPE.

In [75]:
X_tr = np.asarray(X_train, dtype=float)
y_tr = np.asarray(y_train, dtype=float)
X_tr_design = np.hstack([np.ones((X_tr.shape[0], 1)), X_tr])

# вычисление весов 
w = np.linalg.inv(X_tr_design.T @ X_tr_design) @ X_tr_design.T @ y_tr

# предсказание на тесте
X_te = np.asarray(X_test, dtype=float)
X_te_design = np.hstack([np.ones((X_te.shape[0], 1)), X_te])
y_pred_lr = X_te_design @ w

MAPE(y_test, y_pred_lr)

31.75846865524344